In [4]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import os
from pathlib import Path

def convert_height_to_inches(height_str):
    """Converts height from 'feet-inches' string format to total inches."""
    try:
        feet, inches = map(int, height_str.split('-'))
        return feet * 12 + inches
    except (ValueError, AttributeError):
        return np.nan

def get_dataset_config(filepath):
    """Returns configuration for specific datasets."""
    config = {
        'target_column': None,
        'exclude_columns': [],
        'special_handling': None
    }
    
    filename = Path(filepath).name.lower()
    
    if 'airbnb' in filename:
        config['target_column'] = 'room_type'
    elif 'airlines' in filename:
        config['target_column'] = 'satisfaction'
        config['exclude_columns'] = ['Unnamed: 0', 'id']
    elif 'breast_cancer' in filename:
        config['target_column'] = 'diagnosis'
        config['exclude_columns'] = ['id']
    elif 'diabetes' in filename:
        config['target_column'] = 'Outcome'
        config['exclude_columns'] = ['Outcome']
    elif 'digit' in filename:
        config['target_column'] = 'label'
        config['exclude_columns'] = ['label']
    elif 'happiness' in filename:
        config['exclude_columns'] = ['Overall rank']
    elif 'heart_disease' in filename:
        config['target_column'] = 'target'
        config['exclude_columns'] = ['id', 'target']
    elif 'housing' in filename:
        config['target_column'] = 'ocean_proximity'
    elif 'insurance' in filename:
        config['special_handling'] = 'insurance'
    elif 'iris' in filename:
        config['target_column'] = 'Species'
        config['exclude_columns'] = ['Id']
    elif 'mall_customers' in filename:
        config['exclude_columns'] = ['CustomerID']
    elif 'netflix' in filename:
        config['target_column'] = 'type'
        config['special_handling'] = 'netflix'
    elif 'player_data' in filename:
        config['target_column'] = 'position'
        config['special_handling'] = 'player_data'
    elif 'student' in filename:
        config['target_column'] = 'school'
    elif 'telco' in filename or 'churn' in filename:
        config['target_column'] = 'Churn'
        config['exclude_columns'] = ['customerID', 'SeniorCitizen']
        config['special_handling'] = 'telco'
    elif 'titanic' in filename:
        config['target_column'] = 'Survived'
        config['exclude_columns'] = ['PassengerId', 'Survived', 'Pclass']
    elif 'employee' in filename or 'attrition' in filename:
        config['target_column'] = 'Attrition'
        config['exclude_columns'] = ['EmployeeCount', 'EmployeeNumber', 'StandardHours']
    elif 'weather' in filename:
        config['target_column'] = 'Precip Type'
    elif 'wine' in filename:
        config['target_column'] = 'quality'
        config['exclude_columns'] = ['quality']
    elif 'youtube' in filename or 'spam' in filename:
        config['special_handling'] = 'youtube'
    
    return config

def calculate_problem_severity(missing_pct, imbalance_ratio, num_outliers, num_skewed):
    """
    Calculates a severity score based on problem intensity.
    
    Returns:
        float: Severity score (0-100)
    """
    severity = 0
    
    # Missing values severity (0-30 points)
    if missing_pct > 30:
        severity += 30
    elif missing_pct > 15:
        severity += 20
    elif missing_pct > 5:
        severity += 10
    
    # Class imbalance severity (0-25 points)
    if imbalance_ratio:
        if imbalance_ratio > 0.9:
            severity += 25
        elif imbalance_ratio > 0.8:
            severity += 15
        elif imbalance_ratio > 0.7:
            severity += 10
    
    # Outliers severity (0-25 points)
    if num_outliers > 5:
        severity += 25
    elif num_outliers > 3:
        severity += 15
    elif num_outliers > 0:
        severity += 10
    
    # Skewness severity (0-20 points)
    if num_skewed > 5:
        severity += 20
    elif num_skewed > 3:
        severity += 12
    elif num_skewed > 0:
        severity += 6
    
    return severity

def analyze_dataset(filepath):
    """
    Analyzes a dataset for quality issues and returns comprehensive metadata.
    
    Args:
        filepath (str): Path to the CSV file
        
    Returns:
        dict: Dictionary containing analysis results
    """
    print(f"Analyzing {filepath}...")
    
    try:
        # Try different encodings
        try:
            df = pd.read_csv(filepath)
        except UnicodeDecodeError:
            df = pd.read_csv(filepath, encoding='latin-1')
    except Exception as e:
        print(f"  ❌ Could not read {filepath}. Error: {e}")
        return None
    
    # Get dataset configuration
    config = get_dataset_config(filepath)
    
    # Remove unnamed columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    
    # Special handling for specific datasets
    if config['special_handling'] == 'player_data' and 'height' in df.columns:
        df['height_inches'] = df['height'].apply(convert_height_to_inches)
    
    if config['special_handling'] == 'telco' and 'TotalCharges' in df.columns:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    
    # Identify numerical features
    numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Apply special handling for specific datasets
    if config['special_handling'] == 'insurance':
        numerical_features = [c for c in ['age', 'bmi', 'children', 'charges'] if c in df.columns]
    elif config['special_handling'] == 'netflix':
        numerical_features = [c for c in ['release_year'] if c in df.columns]
    elif config['special_handling'] == 'player_data':
        numerical_features = [c for c in ['year_start', 'year_end', 'weight', 'height_inches'] if c in df.columns]
    elif config['special_handling'] == 'youtube':
        # For YouTube datasets, we focus on text content analysis
        numerical_features = []
    else:
        # Remove excluded columns
        numerical_features = [c for c in numerical_features if c not in config['exclude_columns']]
    
    # --- 1. Missing Values Check (>5%) ---
    missing_values = df.isnull().sum() / len(df) * 100
    missing_issue = (missing_values > 5).any()
    missing_percentage = missing_values.max() if len(missing_values) > 0 else 0
    
    # --- 2. Class Imbalance Check (>70%) ---
    imbalance_issue = False
    imbalance_ratio = None
    if config['target_column'] and config['target_column'] in df.columns:
        class_distribution = df[config['target_column']].value_counts(normalize=True)
        if not class_distribution.empty:
            imbalance_ratio = class_distribution.max()
            if imbalance_ratio > 0.7:
                imbalance_issue = True
    
    # --- 3. Outliers Check ---
    outliers_issue = False
    outlier_features = []
    
    # Check for YouTube datasets (text-based outliers)
    if config['special_handling'] == 'youtube' and 'CONTENT' in df.columns:
        df['CONTENT'] = df['CONTENT'].astype(str)
        comment_lengths = df['CONTENT'].str.len()
        if comment_lengths.mean() > 0 and (comment_lengths.max() > comment_lengths.mean() * 5):
            outliers_issue = True
            outlier_features.append('CONTENT')
    
    # Check numerical features for outliers
    for col in numerical_features:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            if IQR > 0:
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
                if outlier_count > 0:
                    outliers_issue = True
                    outlier_features.append(col)
    
    # --- 4. Skewness and Kurtosis Check ---
    skew_kurtosis_issue = False
    skewed_features = []
    for col in numerical_features:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            feature_data = df[col].dropna()
            if len(feature_data) > 0:
                feature_skew = skew(feature_data)
                feature_kurt = kurtosis(feature_data)
                if abs(feature_skew) > 1 or abs(feature_kurt) > 3:
                    skew_kurtosis_issue = True
                    skewed_features.append(col)
    
    # --- Calculate Quality Metrics ---
    problem_count = sum([
        missing_issue,
        imbalance_issue,
        outliers_issue,
        skew_kurtosis_issue
    ])
    
    # Calculate severity score
    severity_score = calculate_problem_severity(
        missing_percentage,
        imbalance_ratio,
        len(outlier_features),
        len(skewed_features)
    )
    
    # Determine quality label based on BOTH problem count AND severity
    # Bad if: 3+ problems OR severity >= 50 OR (2+ problems AND severity >= 35)
    if problem_count >= 3 or severity_score >= 50 or (problem_count >= 2 and severity_score >= 35):
        quality_label = 'Bad'
    else:
        quality_label = 'Good'
    
    # --- Prepare Results ---
    results = {
        'dataset_name': Path(filepath).name,
        'quality_label': quality_label,
        'problem_count': problem_count,
        'severity_score': round(severity_score, 2),
        'missing_values_issue': missing_issue,
        'class_imbalance_issue': imbalance_issue,
        'outliers_issue': outliers_issue,
        'skew_kurtosis_issue': skew_kurtosis_issue,
        'num_rows': len(df),
        'num_columns': len(df.columns),
        'missing_percentage': round(missing_percentage, 2),
        'imbalance_ratio': round(imbalance_ratio, 3) if imbalance_ratio else None,
        'num_outlier_features': len(outlier_features),
        'num_skewed_features': len(skewed_features)
    }
    
    print(f"  ✓ {results['dataset_name']}: {quality_label} (Problems: {problem_count}, Severity: {severity_score:.1f})")
    
    return results

def generate_metadata(search_directory='.', output_file='metadata.csv', file_pattern='*.csv'):
    """
    Generates comprehensive metadata for all datasets in a directory.
    
    Args:
        search_directory (str): Directory to search for CSV files
        output_file (str): Name of the output metadata file
        file_pattern (str): Pattern to match CSV files
    """
    print("=" * 70)
    print("DATASET METADATA GENERATOR - BINARY CLASSIFICATION")
    print("=" * 70)
    
    # Find all CSV files in directory
    search_path = Path(search_directory)
    all_csv_files = list(search_path.glob(file_pattern))
    
    # Exclude the output metadata file itself
    csv_files = [f for f in all_csv_files if f.name != output_file]
    
    if not csv_files:
        print(f"\n⚠️  No CSV files found in '{search_directory}'")
        return
    
    print(f"\nFound {len(csv_files)} dataset(s) to analyze:\n")
    
    # Analyze each dataset
    metadata = []
    for csv_file in sorted(csv_files):
        result = analyze_dataset(str(csv_file))
        if result:
            metadata.append(result)
    
    # Create DataFrame
    if not metadata:
        print("\n⚠️  No datasets were successfully analyzed.")
        return
    
    metadata_df = pd.DataFrame(metadata)
    
    # Define column order
    column_order = [
        'dataset_name', 'quality_label', 'problem_count', 'severity_score',
        'missing_values_issue', 'class_imbalance_issue',
        'outliers_issue', 'skew_kurtosis_issue',
        'num_rows', 'num_columns', 'missing_percentage',
        'imbalance_ratio', 'num_outlier_features', 'num_skewed_features'
    ]
    
    # Reorder columns
    metadata_df = metadata_df[column_order]
    
    # Save to CSV
    metadata_df.to_csv(output_file, index=False)
    
    # Print summary
    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)
    print(f"\n✓ Metadata saved to: '{output_file}'")
    print(f"✓ Total datasets analyzed: {len(metadata_df)}")
    print(f"\nQuality Distribution:")
    print(metadata_df['quality_label'].value_counts().to_string())
    
    print(f"\n{'Dataset':<40} {'Quality':<10} {'Problems':<10} {'Severity':<10}")
    print("-" * 70)
    for _, row in metadata_df.iterrows():
        print(f"{row['dataset_name']:<40} {row['quality_label']:<10} {row['problem_count']:<10} {row['severity_score']:<10.1f}")
    
    return metadata_df

if __name__ == '__main__':
    # Generate metadata for all CSV files in current directory
    metadata_df = generate_metadata()
    
    # Optional: Display detailed statistics
    if metadata_df is not None:
        print("\n" + "=" * 70)
        print("DETAILED STATISTICS")
        print("=" * 70)
        print(f"\nAverage problems per dataset: {metadata_df['problem_count'].mean():.2f}")
        print(f"Average severity score: {metadata_df['severity_score'].mean():.2f}")
        print(f"\nGood datasets: {(metadata_df['quality_label'] == 'Good').sum()}")
        print(f"Bad datasets: {(metadata_df['quality_label'] == 'Bad').sum()}")
        print(f"\nDatasets with missing values: {metadata_df['missing_values_issue'].sum()}")
        print(f"Datasets with class imbalance: {metadata_df['class_imbalance_issue'].sum()}")
        print(f"Datasets with outliers: {metadata_df['outliers_issue'].sum()}")
        print(f"Datasets with skewness issues: {metadata_df['skew_kurtosis_issue'].sum()}")

DATASET METADATA GENERATOR - BINARY CLASSIFICATION

Found 202 dataset(s) to analyze:

Analyzing adult.csv...
  ✓ adult.csv: Good (Problems: 2, Severity: 21.0)
Analyzing adzuna_global_job_listings_2025.csv...
  ✓ adzuna_global_job_listings_2025.csv: Bad (Problems: 3, Severity: 57.0)
Analyzing agents_stats.csv...
  ✓ agents_stats.csv: Bad (Problems: 2, Severity: 45.0)
Analyzing ai_job_market.csv...
  ✓ ai_job_market.csv: Good (Problems: 0, Severity: 0.0)
Analyzing air_quality_global.csv...
  ✓ air_quality_global.csv: Good (Problems: 2, Severity: 16.0)
Analyzing airbnb.csv...
  ✓ airbnb.csv: Bad (Problems: 3, Severity: 65.0)
Analyzing airlines_test.csv...
  ✓ airlines_test.csv: Good (Problems: 2, Severity: 21.0)
Analyzing airlines_train.csv...
  ✓ airlines_train.csv: Good (Problems: 2, Severity: 21.0)
Analyzing all_stocks_5yr.csv...
  ✓ all_stocks_5yr.csv: Good (Problems: 2, Severity: 27.0)
Analyzing amazon_all_electronics_data.csv...
  ✓ amazon_all_electronics_data.csv: Good (Problems: 2

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def create_quality_matrix(metadata_file='metadata.csv', output_file='quality_matrix.csv'):
    """
    Creates a quality_matrix.csv free from data leakage.
    
    Removes:
    - problem_count (directly used in labeling logic)
    - Individual problem flags (missing_values_issue, class_imbalance_issue, etc.)
    - Detailed metrics that reveal problem existence (num_outlier_features, num_skewed_features)
    
    Keeps:
    - severity_score (aggregate metric, useful for model)
    - Raw measurements (num_rows, num_columns, missing_percentage, imbalance_ratio)
    - dataset_name and quality_label (identifier and target)
    """
    
    print("=" * 70)
    print("QUALITY MATRIX GENERATOR - DATA LEAKAGE REMOVAL")
    print("=" * 70)
    
    # Load metadata
    try:
        df = pd.read_csv(metadata_file)
        print(f"\n✓ Loaded metadata from '{metadata_file}'")
        print(f"  Total datasets: {len(df)}")
    except FileNotFoundError:
        print(f"\n❌ Error: '{metadata_file}' not found!")
        return None
    
    # Define leakage columns to remove
    leakage_columns = [
        'problem_count',              # Directly used in labeling
        'missing_values_issue',       # Binary flag revealing problem
        'class_imbalance_issue',      # Binary flag revealing problem
        'outliers_issue',             # Binary flag revealing problem
        'skew_kurtosis_issue',        # Binary flag revealing problem
        'num_outlier_features',       # Reveals outlier problem intensity
        'num_skewed_features'         # Reveals skewness problem intensity
    ]
    
    # Keep only non-leakage columns
    keep_columns = [
        'dataset_name',          # Identifier
        'quality_label',         # Target variable
        'num_rows',              # Raw dimension
        'num_columns',           # Raw dimension
        'severity_score',        # Aggregate metric (kept as requested)
        'missing_percentage',    # Raw measurement
        'imbalance_ratio'        # Raw measurement
    ]
    
    # Verify all keep columns exist
    missing_cols = [col for col in keep_columns if col not in df.columns]
    if missing_cols:
        print(f"\n⚠️  Warning: Missing columns: {missing_cols}")
        keep_columns = [col for col in keep_columns if col in df.columns]
    
    # Create quality matrix
    quality_matrix = df[keep_columns].copy()
    
    # Add engineered features to improve model performance
    print(f"\n{'FEATURE ENGINEERING:':<40}")
    print("-" * 70)
    
    # 1. Dataset size feature (rows × columns)
    quality_matrix['dataset_size'] = quality_matrix['num_rows'] * quality_matrix['num_columns']
    print(f"  ✓ Created 'dataset_size' (rows × columns)")
    
    # 2. Data sparsity indicator (missing_percentage normalized)
    quality_matrix['sparsity_score'] = quality_matrix['missing_percentage'] / 100
    print(f"  ✓ Created 'sparsity_score' (normalized missing %)")
    
    # 3. Aspect ratio (shape of dataset)
    quality_matrix['aspect_ratio'] = quality_matrix['num_rows'] / (quality_matrix['num_columns'] + 1)
    print(f"  ✓ Created 'aspect_ratio' (rows/columns)")
    
    # 4. Log-transformed features for skewed distributions
    quality_matrix['log_rows'] = np.log1p(quality_matrix['num_rows'])
    quality_matrix['log_columns'] = np.log1p(quality_matrix['num_columns'])
    print(f"  ✓ Created 'log_rows' and 'log_columns' (log transforms)")
    
    # 5. Severity-to-size ratio (severity impact relative to dataset size)
    quality_matrix['severity_per_1k_rows'] = quality_matrix['severity_score'] / (quality_matrix['num_rows'] / 1000 + 1)
    print(f"  ✓ Created 'severity_per_1k_rows' (severity normalized by size)")
    
    # 6. Imbalance severity interaction (if imbalance exists)
    quality_matrix['imbalance_severity'] = quality_matrix['imbalance_ratio'].fillna(0) * quality_matrix['severity_score']
    print(f"  ✓ Created 'imbalance_severity' (interaction term)")
    
    # 7. Missing-severity interaction
    quality_matrix['missing_severity'] = quality_matrix['missing_percentage'] * quality_matrix['severity_score'] / 100
    print(f"  ✓ Created 'missing_severity' (interaction term)")
    
    # 8. Data quality risk score (composite metric)
    # Combines multiple factors without revealing individual problems
    quality_matrix['quality_risk'] = (
        quality_matrix['sparsity_score'] * 30 +
        quality_matrix['imbalance_ratio'].fillna(0.5) * 25 +
        (quality_matrix['severity_score'] / 100) * 45
    )
    print(f"  ✓ Created 'quality_risk' (composite risk metric)")
    
    # Update keep_columns list with engineered features
    engineered_features = [
        'dataset_size', 'sparsity_score', 'aspect_ratio', 
        'log_rows', 'log_columns', 'severity_per_1k_rows',
        'imbalance_severity', 'missing_severity', 'quality_risk'
    ]
    
    # Print removal summary
    print(f"\n{'REMOVED COLUMNS (Data Leakage):':<40}")
    print("-" * 70)
    for col in leakage_columns:
        if col in df.columns:
            print(f"  ❌ {col}")
    
    print(f"\n{'KEPT COLUMNS (No Leakage):':<40}")
    print("-" * 70)
    for col in keep_columns:
        print(f"  ✓ {col}")
    
    # Save quality matrix
    quality_matrix.to_csv(output_file, index=False)
    
    print("\n" + "=" * 70)
    print("QUALITY MATRIX CREATED")
    print("=" * 70)
    print(f"\n✓ Saved to: '{output_file}'")
    print(f"✓ Total datasets: {len(quality_matrix)}")
    print(f"✓ Base features: {len(keep_columns) - 2}")  # Exclude name and label
    print(f"✓ Engineered features: {len(engineered_features)}")
    print(f"✓ Total features: {len(quality_matrix.columns) - 2}")  # Exclude name and label
    
    # Display preview
    print(f"\nPreview of quality_matrix.csv:")
    print("-" * 70)
    print(quality_matrix.head(10).to_string(index=False))
    
    # Data integrity checks
    print("\n" + "=" * 70)
    print("DATA INTEGRITY CHECKS")
    print("=" * 70)
    
    # Check for missing values in features
    missing_check = quality_matrix.drop(columns=['dataset_name', 'quality_label']).isnull().sum()
    print(f"\nMissing values per feature:")
    for col, count in missing_check.items():
        status = "⚠️" if count > 0 else "✓"
        print(f"  {status} {col}: {count}")
    
    # Check class distribution
    print(f"\nClass distribution:")
    class_dist = quality_matrix['quality_label'].value_counts()
    for label, count in class_dist.items():
        pct = count / len(quality_matrix) * 100
        print(f"  {label}: {count} ({pct:.1f}%)")
    
    # Feature statistics
    print(f"\nFeature statistics:")
    numeric_cols = ['num_rows', 'num_columns', 'severity_score', 'missing_percentage', 
                   'imbalance_ratio', 'dataset_size', 'quality_risk']
    stats = quality_matrix[numeric_cols].describe().loc[['min', 'mean', 'max']]
    print(stats.to_string())
    
    # Check for infinite or invalid values
    print(f"\nData validation:")
    inf_check = np.isinf(quality_matrix.select_dtypes(include=[np.number])).sum()
    if inf_check.sum() > 0:
        print(f"  ⚠️  Infinite values detected:")
        for col, count in inf_check[inf_check > 0].items():
            print(f"     {col}: {count}")
    else:
        print(f"  ✓ No infinite values")
    
    nan_check = quality_matrix.select_dtypes(include=[np.number]).isnull().sum()
    if nan_check.sum() > 0:
        print(f"  ⚠️  NaN values detected (will be handled during training)")
    else:
        print(f"  ✓ No NaN values")
    
    print("\n" + "=" * 70)
    print("✓ Quality matrix is ready for model training!")
    print("=" * 70)
    
    return quality_matrix


if __name__ == '__main__':
    # Generate quality matrix from metadata
    quality_matrix = create_quality_matrix()
    
    # Optional: Verify no leakage by checking correlation with target
    if quality_matrix is not None:
        print("\n" + "=" * 70)
        print("LEAKAGE VERIFICATION")
        print("=" * 70)
        
        # Encode quality label for correlation
        quality_encoded = (quality_matrix['quality_label'] == 'Bad').astype(int)
        
        # Calculate correlations
        numeric_features = ['num_rows', 'num_columns', 'severity_score', 
                           'missing_percentage', 'imbalance_ratio',
                           'dataset_size', 'quality_risk', 'sparsity_score',
                           'severity_per_1k_rows']
        
        print("\nFeature correlations with 'Bad' label:")
        print("(Values close to 1.0 indicate potential leakage)")
        print("-" * 70)
        
        correlations = []
        for feature in numeric_features:
            # Handle missing values
            valid_data = quality_matrix[[feature]].dropna()
            valid_labels = quality_encoded[valid_data.index]
            
            if len(valid_data) > 0:
                corr = valid_data[feature].corr(valid_labels)
                correlations.append((feature, corr))
                status = "⚠️" if abs(corr) > 0.9 else "✓"
                print(f"  {status} {feature:<30} {corr:>7.4f}")
        
        # Sort by absolute correlation
        correlations.sort(key=lambda x: abs(x[1]), reverse=True)
        
        print(f"\nTop 3 most predictive features:")
        for i, (feature, corr) in enumerate(correlations[:3], 1):
            print(f"  {i}. {feature}: {corr:.4f}")
        
        max_corr = max(abs(c[1]) for c in correlations)
        if max_corr < 0.9:
            print(f"\n✓ No severe leakage detected (max correlation: {max_corr:.4f})")
        else:
            print(f"\n⚠️  Potential leakage detected (max correlation: {max_corr:.4f})")

QUALITY MATRIX GENERATOR - DATA LEAKAGE REMOVAL

✓ Loaded metadata from 'metadata.csv'
  Total datasets: 200

FEATURE ENGINEERING:                    
----------------------------------------------------------------------
  ✓ Created 'dataset_size' (rows × columns)
  ✓ Created 'sparsity_score' (normalized missing %)
  ✓ Created 'aspect_ratio' (rows/columns)
  ✓ Created 'log_rows' and 'log_columns' (log transforms)
  ✓ Created 'severity_per_1k_rows' (severity normalized by size)
  ✓ Created 'imbalance_severity' (interaction term)
  ✓ Created 'missing_severity' (interaction term)
  ✓ Created 'quality_risk' (composite risk metric)

REMOVED COLUMNS (Data Leakage):         
----------------------------------------------------------------------
  ❌ problem_count
  ❌ missing_values_issue
  ❌ class_imbalance_issue
  ❌ outliers_issue
  ❌ skew_kurtosis_issue
  ❌ num_outlier_features
  ❌ num_skewed_features

KEPT COLUMNS (No Leakage):              
------------------------------------------------